# Step 7: The whole agent with a different router
### Neeraj, MMAI 894 (router fine-tuning)

**Why:** a router can look fine on its own and still fail the customer: Kawtar's fine-tuned router reached 40% task success in the whole agent against 70% with Haiku, because handoffs went to the wrong team. So the routers from the final run go through the **team's own end-to-end test**, unchanged: the same agent, the same specialist desks, the same rule checks and the same AI judge (GPT-6.2 Sol, checked against two people), on the 40 test chats and the 31 fresh chats.

| Setup | Router |
|---|---|
| `full_agent` | Haiku, the team's recommended agent (replayed from the team's recorded calls: no new cost) |
| `full_agent_router_ft_balanced` | fine-tuned, balanced (LoRA + attack answer + own unsure line) |
| `full_agent_router_ft_safest` | fine-tuned, safest (full fine-tuning + attack answer) |
| `full_agent_router_jev2` | Jev with the same information as Haiku (full conversation, as Haiku reads it in the agent) |
| `full_agent_router_finetuned` | Kawtar's fine-tuned router, test chats only (her recording; replayed, for continuity with her 40%) |

**How:** the team's harness already replays a router's recorded answers (`RecordedRouter`, the way it ran Kawtar's model). The final run (notebook 06) saved our routers' answers in that format. This notebook clones the team repo at the commit Libo's results come from (`0cc6208`), points the harness at our recordings, and changes nothing else.

**What you need to do:** Accelerator **None**, Internet **On**, the `OPENROUTER_API_KEY` secret ticked. Run all cells: about 20 to 30 minutes. The desks and the judge make new calls wherever a router sends a chat somewhere new: expect a few dollars (the run stops itself at $10). Download `results_07.zip` and the notebook into `results/step07`.

Everything shown is computed from the runs; this text holds no results.

## Step 0: The team's code (pinned) and our recordings

In [ ]:
import os
import shutil
import subprocess
import sys

TEAM, OURS = "/kaggle/working/team", "/kaggle/working/ours"
for d in (TEAM, OURS):
    if os.path.exists(d):
        shutil.rmtree(d)
!git clone -q https://github.com/Salikawtar/northstar-agent {TEAM}
!git -C {TEAM} checkout -q 0cc6208
!git clone -q https://github.com/neerajmehla28/northstar-finetuning {OURS}
!pip install -q faker openpyxl pydantic openai python-dotenv pyyaml
!git -C {TEAM} log --oneline -1
!git -C {OURS} log --oneline -1

REC = f"{OURS}/results/step06/recordings"
ROUTERS = {"full_agent_router_ft_balanced": (f"{REC}/router_finetuned-balanced_test_and_fresh.csv", "router: fine-tuned, balanced (LoRA + attack answer + own unsure line)"),
           "full_agent_router_ft_safest": (f"{REC}/router_finetuned-safest_test_and_fresh.csv", "router: fine-tuned, safest (full fine-tuning + attack answer)"),
           "full_agent_router_jev2": (f"{REC}/router_jev2_full_conversation_test_and_fresh.csv", "router: Jev with the same information as Haiku")}
for name, (path, _) in ROUTERS.items():
    assert os.path.exists(path), f"missing recording {path}: have the final run's results been pushed?"
print("recordings found:", list(ROUTERS))

## Step 1: Load the team's harness and register our routers
Only the router changes: `build_router` returns a `RecordedRouter` with our recording for our setups and the team's own router for every other setup.

In [ ]:
from pathlib import Path

import pandas as pd
API_KEY = None
try:                                    # Kaggle secret; without one, the recorded calls are replayed for free
    from kaggle_secrets import UserSecretsClient
    API_KEY = UserSecretsClient().get_secret("OPENROUTER_API_KEY")
except Exception:
    pass
if API_KEY:
    os.environ["OPENROUTER_API_KEY"] = API_KEY
saved = f"{OURS}/results/step07/llm_cache_new"          # the calls this test made when it was run, saved in our repo
if os.path.exists(saved):
    shutil.copytree(saved, f"{TEAM}/data/llm_cache", dirs_exist_ok=True)
os.chdir(TEAM)
sys.path.insert(0, f"{TEAM}/src")

from northstar.dms.store import DealerSystem
from northstar.evaluation import harness as H
from northstar.evaluation import metrics
from northstar.evaluation import setups as S
from northstar.evaluation.__main__ import evaluate_setup
from northstar.evaluation.grader import ModelGrader
from northstar.evaluation.rules import RuleGrader
from northstar.llm import LLMClient

_team_build_router = H.build_router
def build_router(setup, client):
    if setup.name in ROUTERS:
        return S.RecordedRouter(Path(ROUTERS[setup.name][0]), name=setup.name)
    return _team_build_router(setup, client)
H.build_router = build_router
for name, (_, change) in ROUTERS.items():
    S.SETUPS[name] = S.Setup(name, change, router_model="finetuned")

client = LLMClient(mode="live" if API_KEY else "replay", budget_usd=10.0)
print("mode:", client.mode)
grader, rules = ModelGrader(client), RuleGrader(DealerSystem())
(Path(TEAM) / "src/northstar/evaluation/results/grade_records").mkdir(parents=True, exist_ok=True)
print("setups:", ["full_agent"] + list(ROUTERS), "| grader:", grader.model)

## Step 2: Run and grade every setup on the test chats, then on the fresh chats
`full_agent` comes first: it should replay entirely from the team's recorded calls (new money spent stays at $0), which also checks that this copy of the team's harness reproduces Libo's numbers. Rerunning the cell reuses every recorded call.

In [ ]:
import time
frames = []
def setups_for(split):
    return ["full_agent"] + list(ROUTERS) + (["full_agent_router_finetuned"] if split == "test" else [])

for split in ("test", "fresh"):
    for name in setups_for(split):
        t0, before = time.time(), client.usage.spent_usd
        df = evaluate_setup(name, split, client, grader, rules)
        df["split"] = split
        frames.append(df)
        print(f"{split:5s} {name:31s} task success {df['success'].mean():.3f} | new money ${client.usage.spent_usd - before:.2f} | {time.time() - t0:.0f}s", flush=True)
per_chat = pd.concat(frames, ignore_index=True)
print(f"Total new money spent: ${client.usage.spent_usd:.2f}")

## Step 3: Results
The team's own headline numbers for each setup (task success = all five rubric questions pass and no rule check fails), plus how many handoffs reached the right team, and an exact paired test against the Haiku agent on the same chats.

In [ ]:
rows, pairs = [], []
for split in ("test", "fresh"):
    d = per_chat[per_chat["split"] == split]
    base = d[d["setup"] == "full_agent"].set_index("id")["success"]
    for name in setups_for(split):
        g = d[d["setup"] == name]
        s = metrics.summarize(g)
        made = g[g["gold_escalation"] & g["escalated"]]
        rows.append(dict(split=split, setup=name, chats=s["n"], task_success=s["task_success"],
                         handoffs_needed=int(g["gold_escalation"].sum()), missed_handoffs=s["missed_handoffs"],
                         handoffs_right_team=int((made["queue_ok"] != False).sum()), extra_handoffs=s["extra_handoffs"],
                         policy_failures=s["policy_failures"], invented_facts=s["invented_facts"], wrong_commitments=s["wrong_commitments"],
                         handled_without_person_and_correct=s["handled_without_person_and_correct"],
                         latency_s_median=s["latency_s_median"], cost_usd_per_1000_chats=s["cost_usd_per_1000_chats"]))
        if name != "full_agent":
            other = g.set_index("id")["success"].reindex(base.index)
            t = metrics.mcnemar_exact(base.values, other.values)
            pairs.append(dict(split=split, setup=name, versus="full_agent (Haiku)", only_haiku_agent_right=t["a_only"],
                              only_this_router_right=t["b_only"], p_value=t["p_value"]))
summary = pd.DataFrame(rows)
paired = pd.DataFrame(pairs)
OUT = "/kaggle/working/results_07"
os.makedirs(OUT, exist_ok=True)
summary.to_csv(f"{OUT}/agent_summary.csv", index=False)
paired.to_csv(f"{OUT}/agent_paired_vs_haiku.csv", index=False)
per_chat.drop(columns=["case"], errors="ignore").to_csv(f"{OUT}/agent_per_chat.csv", index=False)
display(summary)
display(paired)

## Step 4: Does the Haiku agent reproduce the team's numbers?
Libo's tables (`src/northstar/evaluation/results/headline*.csv` in the team repo) against this run's `full_agent`. They should match exactly, since every call is replayed.

In [ ]:
for split, f in (("test", "headline.csv"), ("fresh", "headline_fresh.csv")):
    team = pd.read_csv(Path(TEAM) / "src/northstar/evaluation/results" / f)
    print(split, "| team's file:")
    display(team.head(3))
    print(split, "| this run, full_agent:", summary[(summary.split == split) & (summary.setup == "full_agent")][["task_success", "missed_handoffs", "extra_handoffs"]].to_dict("records"))

## Step 5: Save the results
Download `results_07.zip` and the notebook (File -> Download Notebook) into `results/step07`. The zip also holds the judge's records and the team repo's recorded calls made in this run, so it replays for free.

In [ ]:
shutil.copytree(f"{TEAM}/src/northstar/evaluation/results/grade_records", f"{OUT}/grade_records", dirs_exist_ok=True)
new_cache = subprocess.run(["git", "-C", TEAM, "ls-files", "--others", "--exclude-standard", "data/llm_cache"], capture_output=True, text=True).stdout.split()
os.makedirs(f"{OUT}/llm_cache_new", exist_ok=True)
for f in new_cache:
    shutil.copy(f"{TEAM}/{f}", f"{OUT}/llm_cache_new/")
print("new recorded calls:", len(new_cache))
shutil.make_archive("/kaggle/working/results_07", "zip", OUT)
print(f"results_07.zip {os.path.getsize('/kaggle/working/results_07.zip') / 1e6:.1f} MB")

## What this run shows, and its limits
- **Shows:** what each router does to the customer in the team's real agent, graded by the team's own rubric and judge: task success, missed and extra handoffs, the right team, policy and fact errors, speed and cost.
- **Limits:**
  - 40 and 31 chats, one run each: wide ranges; a few chats are not proof.
  - The judge agreed with two people at kappa 0.71 and 0.77, but not on rare commitment errors.
  - The fine-tuned routers read the customer's first message (their frozen design); Haiku and Jev read the whole conversation inside the agent.
  - The team's code is pinned at `0cc6208`; later team changes are not included.